In [ ]:
import sys
import subprocess

def ensure_package(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

ensure_package('datasets')
ensure_package('pandas')
ensure_package('numpy')
ensure_package('scikit-learn', 'sklearn')

In [ ]:
import random
import re
import time
from itertools import product

import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option('display.max_colwidth', 200)

In [ ]:
dataset = load_dataset('emotion')
print(dataset)
print('Splits:', list(dataset.keys()))
for split in dataset.keys():
    print(split, len(dataset[split]))

In [ ]:
label_feature = dataset['train'].features['label']
label_names = label_feature.names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}
print('Labels:', id2label)

train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

for df in [train_df, val_df, test_df]:
    df['label_name'] = df['label'].map(id2label)
    df['char_len'] = df['text'].astype(str).str.len()
    df['word_len'] = df['text'].astype(str).str.split().str.len()

print(train_df.head(10))

In [ ]:
print('Train label distribution:')
print(train_df['label_name'].value_counts().sort_index())
print('\nValidation label distribution:')
print(val_df['label_name'].value_counts().sort_index())
print('\nTest label distribution:')
print(test_df['label_name'].value_counts().sort_index())

print('\nText length stats (train):')
print(train_df[['char_len', 'word_len']].describe())

In [ ]:
STOPWORDS = set(ENGLISH_STOP_WORDS)

def simple_lemmatize_token(token):
    t = token
    if len(t) > 4 and t.endswith('ies'):
        return t[:-3] + 'y'
    if len(t) > 5 and t.endswith('ing'):
        base = t[:-3]
        if len(base) >= 2 and base[-1] == base[-2]:
            base = base[:-1]
        return base
    if len(t) > 4 and t.endswith('ed'):
        base = t[:-2]
        if len(base) >= 2 and base[-1] == base[-2]:
            base = base[:-1]
        return base
    if len(t) > 4 and t.endswith('es'):
        return t[:-2]
    if len(t) > 3 and t.endswith('s') and not t.endswith('ss'):
        return t[:-1]
    return t

def preprocess_text(text, lowercase=True, remove_stopwords=False, lemma_variant='none'):
    text = str(text).strip()
    if lowercase:
        text = text.lower()
    text = re.sub(r"[^\w\s']", ' ', text)
    tokens = text.split()
    if remove_stopwords:
        tokens = [tok for tok in tokens if tok not in STOPWORDS]
    if lemma_variant == 'simple':
        tokens = [simple_lemmatize_token(tok) for tok in tokens]
    elif lemma_variant == 'alpha_only':
        tokens = [tok for tok in tokens if tok.isalpha()]
    return ' '.join(tokens)

sample_pre = pd.DataFrame({
    'original': train_df['text'].head(5),
    'default_clean': train_df['text'].head(5).map(lambda x: preprocess_text(x, lowercase=True, remove_stopwords=False, lemma_variant='none')),
    'stopwords_removed': train_df['text'].head(5).map(lambda x: preprocess_text(x, lowercase=True, remove_stopwords=True, lemma_variant='none')),
    'simple_lemma': train_df['text'].head(5).map(lambda x: preprocess_text(x, lowercase=True, remove_stopwords=False, lemma_variant='simple'))
})
print(sample_pre.to_string(index=False))

In [ ]:
ablation_configs = []

lowercase_options = [True, False]
stopword_options = [False, True]
ngram_options = [(1, 1), (1, 2)]
min_df_options = [1, 2]
lemma_options = ['none', 'simple', 'alpha_only']

for lowercase, remove_stopwords, ngram_range, min_df, lemma_variant in product(
    lowercase_options,
    stopword_options,
    ngram_options,
    min_df_options,
    lemma_options
):
    config_name = f"lc={lowercase}|sw={remove_stopwords}|ng={ngram_range}|min_df={min_df}|lemma={lemma_variant}"
    ablation_configs.append({
        'config_name': config_name,
        'lowercase': lowercase,
        'remove_stopwords': remove_stopwords,
        'ngram_range': ngram_range,
        'min_df': min_df,
        'lemma_variant': lemma_variant
    })

print('Total ablation configs:', len(ablation_configs))
print(pd.DataFrame(ablation_configs).head(10).to_string(index=False))

In [ ]:
X_train_raw = train_df['text'].tolist()
y_train = train_df['label'].tolist()
X_val_raw = val_df['text'].tolist()
y_val = val_df['label'].tolist()
X_test_raw = test_df['text'].tolist()
y_test = test_df['label'].tolist()

ablation_results = []
best_val_f1 = -1.0
best_model = None
best_config = None
best_test_preds = None
best_val_preds = None
best_processed = None

for i, cfg in enumerate(ablation_configs, start=1):
    print(f"[{i}/{len(ablation_configs)}] Running: {cfg['config_name']}")
    
    X_train = [preprocess_text(x, cfg['lowercase'], cfg['remove_stopwords'], cfg['lemma_variant']) for x in X_train_raw]
    X_val = [preprocess_text(x, cfg['lowercase'], cfg['remove_stopwords'], cfg['lemma_variant']) for x in X_val_raw]
    X_test = [preprocess_text(x, cfg['lowercase'], cfg['remove_stopwords'], cfg['lemma_variant']) for x in X_test_raw]
    
    model = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=cfg['ngram_range'],
            min_df=cfg['min_df'],
            max_df=0.95,
            sublinear_tf=True
        )),
        ('clf', LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=None))
    ])
    
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train
    
    start_val = time.time()
    val_preds = model.predict(X_val)
    val_infer_time = time.time() - start_val
    
    start_test = time.time()
    test_preds = model.predict(X_test)
    test_infer_time = time.time() - start_test
    
    val_acc = accuracy_score(y_val, val_preds)
    val_f1 = f1_score(y_val, val_preds, average='macro')
    test_acc = accuracy_score(y_test, test_preds)
    test_f1 = f1_score(y_test, test_preds, average='macro')
    
    vocab_size = len(model.named_steps['tfidf'].vocabulary_)
    
    row = {
        'config_name': cfg['config_name'],
        'lowercase': cfg['lowercase'],
        'remove_stopwords': cfg['remove_stopwords'],
        'ngram_range': str(cfg['ngram_range']),
        'min_df': cfg['min_df'],
        'lemma_variant': cfg['lemma_variant'],
        'vocab_size': vocab_size,
        'val_accuracy': val_acc,
        'val_macro_f1': val_f1,
        'test_accuracy': test_acc,
        'test_macro_f1': test_f1,
        'train_time_sec': train_time,
        'val_inference_sec': val_infer_time,
        'test_inference_sec': test_infer_time
    }
    ablation_results.append(row)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model = model
        best_config = cfg
        best_test_preds = test_preds
        best_val_preds = val_preds
        best_processed = {
            'X_train': X_train,
            'X_val': X_val,
            'X_test': X_test
        }

results_df = pd.DataFrame(ablation_results)
results_df = results_df.sort_values(['val_macro_f1', 'val_accuracy', 'test_macro_f1'], ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
print('Best configuration:')
print(best_config)
print('\nTop 10 ablation results:')
print(results_df.head(10).to_string(index=False))

baseline_row = results_df[
    (results_df['lowercase'] == True) &
    (results_df['remove_stopwords'] == False) &
    (results_df['ngram_range'] == str((1, 2))) &
    (results_df['min_df'] == 2) &
    (results_df['lemma_variant'] == 'none')
]

if len(baseline_row) > 0:
    baseline_row = baseline_row.iloc[0]
    best_row = results_df.iloc[0]
    comparison_df = pd.DataFrame([
        {
            'setting': 'baseline_like_original',
            'val_accuracy': baseline_row['val_accuracy'],
            'val_macro_f1': baseline_row['val_macro_f1'],
            'test_accuracy': baseline_row['test_accuracy'],
            'test_macro_f1': baseline_row['test_macro_f1']
        },
        {
            'setting': 'best_ablation',
            'val_accuracy': best_row['val_accuracy'],
            'val_macro_f1': best_row['val_macro_f1'],
            'test_accuracy': best_row['test_accuracy'],
            'test_macro_f1': best_row['test_macro_f1']
        }
    ])
    print('\nBaseline vs best:')
    print(comparison_df.to_string(index=False))

In [ ]:
print('Validation classification report for best configuration:')
print(classification_report(y_val, best_val_preds, target_names=label_names, digits=4))

print('Test classification report for best configuration:')
print(classification_report(y_test, best_test_preds, target_names=label_names, digits=4))

cm = confusion_matrix(y_test, best_test_preds)
cm_df = pd.DataFrame(cm, index=[f'true_{x}' for x in label_names], columns=[f'pred_{x}' for x in label_names])
print('Test confusion matrix for best configuration:')
print(cm_df)

In [ ]:
error_df = test_df[['text', 'label', 'label_name', 'char_len', 'word_len']].copy()
error_df['processed_text'] = best_processed['X_test']
error_df['pred'] = best_test_preds
error_df['pred_name'] = error_df['pred'].map(id2label)
error_df['correct'] = error_df['label'] == error_df['pred']

misclassified = error_df[~error_df['correct']].copy()
print('Total test examples:', len(error_df))
print('Misclassified test examples:', len(misclassified))

print('Top confusion pairs:')
print(misclassified.groupby(['label_name', 'pred_name']).size().sort_values(ascending=False).head(20))

print('\nSample misclassifications:')
print(misclassified[['text', 'processed_text', 'label_name', 'pred_name', 'word_len']].head(25).to_string(index=False))

In [ ]:
def predict_emotion(texts, model, config, id2label):
    cleaned = [
        preprocess_text(t, config['lowercase'], config['remove_stopwords'], config['lemma_variant'])
        for t in texts
    ]
    pred_ids = model.predict(cleaned)
    if hasattr(model, 'predict_proba'):
        probas = model.predict_proba(cleaned)
        confs = probas.max(axis=1)
    else:
        confs = [None] * len(cleaned)
    return pd.DataFrame({
        'text': texts,
        'processed_text': cleaned,
        'pred_label_id': pred_ids,
        'pred_label': [id2label[i] for i in pred_ids],
        'confidence': confs
    })

sample_texts = [
    'i feel amazing and grateful today',
    'i am really upset and angry about what happened',
    'i miss my friends and feel lonely',
    'i am scared about tomorrow',
    'this was such a lovely surprise'
]

print('Best config used for inference:')
print(best_config)
print(predict_emotion(sample_texts, best_model, best_config, id2label).to_string(index=False))